# IID experiment — configurable runner (single / multi)
**Edit the CONFIG cell, run everything below it.** Protocol: noise-BEFORE the front,
best epoch by detection (Pd@Pfa), grad clip only (no wd, no eigen/MAD floors),
checkpoints + raw scores saved per run, resume-safe.
Fronts: `std` | `robust` (median/MAD) | `zca` (no floor).

In [ ]:
!git clone -b rebuttal --depth 1 https://github.com/michaelpiro/final-paper-experiment.git repo
%cd repo
import os, torch
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
assert os.path.exists('repro/data/pavia-u.mat'), 'missing data'

In [ ]:
# ======================= CONFIG — edit me =======================
MODE   = 'single'        # 'single' | 'multi'
NS     = [2048]          # training-pool sizes; paper grid: [20,40,60,100,200,500,1000,2000]
RHOS   = [0.01]          # multi optimum .003 | single optimum .01
SEEDS  = [42, 43, 44]    # paper uses [42,43,44,45,46]
AMP    = None            # amplitude (theta); None -> paper value from config (0.15)
BKG_CLS    = None        # None -> config (single: 2=meadows); classes: 1 asphalt 2 meadows
TARGET_CLS = None        # None -> config (4=trees); 3 gravel 4 trees 5 metal 6 soil 7 bitumen
PFA    = 0.1             # false-alarm rate for the Pd criterion

FRONT  = 'std'           # 'std' | 'robust' | 'zca'
HIDDEN = [128]
EPOCHS = 15000
EVAL_START, EVAL_EVERY = 200, 100
LR     = 5e-4
WD     = 0.0             # weight decay KILLS DART (.835 -> .435) — keep 0
CLIP   = 1.0             # grad-norm clip; the only clip anywhere
BATCH  = 512

OUT       = 'results_iid.json'
CKPT_DIR  = 'ckpt_iid'
SAVE_CKPT = True         # best-epoch weights + raw scores per run
# =================================================================

In [ ]:
# ENGINE — no knobs here; edit CONFIG above and re-run this cell
import copy, json, yaml
import numpy as np, torch
from tqdm import tqdm
from repro.protocols.iid import load_hsi, build_pools, _pd_at_fa, _auc
from repro.core.data import Whitening, plant_targets
from repro.core.models import ScoreNet

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_grad_enabled(True)
os.makedirs(CKPT_DIR, exist_ok=True)

cfg = yaml.safe_load(open(f'repro/configs/iid_{MODE}.yaml'))
cfg['dataset'] = 'repro/data/pavia-u.mat'
if BKG_CLS is not None:    cfg['bkg_cls'] = int(BKG_CLS)
if TARGET_CLS is not None: cfg['target_cls'] = int(TARGET_CLS)
theta = float(cfg['amplitude']) if AMP is None else float(AMP)
data, gt = load_hsi(cfg['dataset'])
bkg, tgt = build_pools(data, gt.flatten(), cfg, MODE)
s = tgt.mean(axis=0).astype(np.float32)
print(f"{MODE}: bkg pool {len(bkg)} px, target cls {cfg['target_cls']} "
      f"({len(tgt)} px), theta={theta}")

def make_front(tr):
    X = np.asarray(tr, np.float64)
    if FRONT == 'robust':
        med = np.median(X, axis=0)
        scale = np.median(np.abs(X - med), axis=0) * 1.4826
        return Whitening(med.astype(np.float32), np.diag(1.0/scale).astype(np.float32))
    if FRONT == 'std':
        return Whitening(X.mean(0).astype(np.float32),
                         np.diag(1.0/X.std(0)).astype(np.float32))
    lam, V = np.linalg.eigh(np.cov(X, rowvar=False))     # zca, raw spectrum
    return Whitening(X.mean(0).astype(np.float32),
                     (V @ np.diag(1.0/np.sqrt(lam)) @ V.T).astype(np.float32))

res = json.load(open(OUT)) if os.path.exists(OUT) else {}
for n in NS:
    for rho in RHOS:
        for seed in SEEDS:
            key = (f"{MODE}_b{cfg['bkg_cls']}t{cfg['target_cls']}_{FRONT}"
                   f"_n{n}_r{rho}_a{theta}_s{seed}")
            if key in res:
                print('skip (done):', key); continue
            rng = np.random.default_rng(seed)
            idx = np.arange(len(bkg)); rng.shuffle(idx)
            shuf = bkg[idx]
            tr = shuf[:n].astype(np.float32)
            te = shuf[-int(cfg['test_size']):].astype(np.float32)
            planted, labels, _ = plant_targets(te, s, theta, cfg['target_fraction'],
                                               model='additive', seed=seed)
            planted = planted.astype(np.float32)
            y = np.asarray(labels)
            D = tr.shape[1]
            sigma = float(np.sqrt(rho * np.asarray(tr, np.float64).var(0).mean()))
            torch.manual_seed(seed)
            net = ScoreNet(D, HIDDEN, 'relu', whitening=make_front(tr)).to(DEVICE)
            opt = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=WD)
            gen = torch.Generator(device=DEVICE); gen.manual_seed(97 * seed)
            X = torch.tensor(tr, device=DEVICE)
            Pt = torch.tensor(planted, device=DEVICE)

            def psi(A):
                out = []
                with torch.no_grad():
                    for i in range(0, len(A), 4096):
                        out.append(net(A[i:i+4096]).cpu().numpy())
                return np.concatenate(out, 0)

            best, curve = {'pd': -1.0}, []
            best_state, best_scores = None, None
            bar = tqdm(range(1, EPOCHS + 1), desc=key, dynamic_ncols=True,
                       mininterval=5.0, ascii=True)
            for ep in bar:
                net.train()
                perm = torch.randperm(n, generator=gen, device=DEVICE)
                for i in range(0, n, BATCH):
                    b = X[perm[i:i+BATCH]]
                    eps = torch.randn(b.shape, generator=gen, device=DEVICE) * sigma
                    loss = ((net(b + eps) + eps / sigma**2)**2).sum(-1).mean()
                    opt.zero_grad(); loss.backward()
                    if CLIP: torch.nn.utils.clip_grad_norm_(net.parameters(), CLIP)
                    opt.step()
                if ep >= EVAL_START and (ep % EVAL_EVERY == 0 or ep == EPOCHS):
                    net.eval()
                    z_tr, z_te = psi(X), psi(Pt)
                    zb = z_tr.mean(0); Cz = np.cov(z_tr, rowvar=False)
                    T = -((z_te - zb) @ s) / np.sqrt(float(s @ Cz @ s))
                    pd = float(_pd_at_fa(y, T, PFA)); auc = float(_auc(y, T))
                    curve.append({'epoch': ep, 'pd': round(pd, 4), 'auc': round(auc, 4)})
                    if pd > best['pd']:
                        best = {'pd': pd, 'auc': auc, 'epoch': ep}
                        if SAVE_CKPT:
                            best_state = copy.deepcopy(
                                {k: v.cpu() for k, v in net.state_dict().items()})
                            best_scores = np.asarray(T, np.float32)
                    bar.set_postfix_str(f'loss={float(loss):.3g} pd={pd:.3f} '
                                        f'best={best["pd"]:.3f}@{best["epoch"]}')
            bar.close()
            if SAVE_CKPT and best_state is not None:
                torch.save({'net': best_state, 'epoch': best['epoch'],
                            'pd': best['pd'], 'auc': best['auc'],
                            'scores': best_scores, 'labels': y,
                            'mode': MODE, 'front': FRONT, 'n': n, 'rho': rho,
                            'theta': theta, 'seed': seed, 'lr': LR, 'wd': WD,
                            'clip': CLIP, 'sigma_raw': sigma,
                            'bkg_cls': cfg['bkg_cls'],
                            'target_cls': cfg['target_cls']},
                           os.path.join(CKPT_DIR, key + '.pt'))
            res[key] = {'pd_best': round(best['pd'], 4),
                        'auc_at_best': round(best['auc'], 4),
                        'best_epoch': best['epoch'], 'pd_final': curve[-1]['pd'],
                        'mode': MODE, 'front': FRONT, 'n': n, 'rho': rho,
                        'theta': theta, 'lr': LR, 'wd': WD, 'clip': CLIP,
                        'batch': BATCH, 'curve': curve}
            json.dump(res, open(OUT, 'w'), indent=1)
            print(f"[{key}] best={best['pd']:.3f}@{best['epoch']} "
                  f"final={curve[-1]['pd']}", flush=True)
print('ALL DONE')

In [ ]:
# Results table: mean +/- std of best-epoch Pd per (n, rho) cell
import json, numpy as np
res = json.load(open(OUT))
rows = {}
for k, v in res.items():
    if v['mode'] != MODE or v['front'] != FRONT: continue
    rows.setdefault((v['n'], v['rho']), []).append(v['pd_best'])
print(f'{MODE} / {FRONT}   (Pd@Pfa={PFA}, best-epoch)')
for (n, rho), vals in sorted(rows.items()):
    print(f'  n={n:5d} rho={rho:<7} {np.mean(vals):.3f} +/- {np.std(vals):.3f}'
          f'  ({len(vals)} seeds)')

In [ ]:
# n-sweep plot vs published lines (only cells present in results are drawn)
import json, numpy as np, matplotlib.pyplot as plt
PUB = {  # published Pd@Pfa=.1, n grid [20,40,60,100,200,500,1000,2000]
 'multi':  {'DART': [.287,.300,.327,.390,.402,.447,.537,.595],
            'AMF':  [.259,.253,.226,.152,.349,.426,.463,.493],
            'LRao': [.329,.367,.465,.501,.509,.565,.561,.553]},
 'single': {'DART': [.539,.492,.500,.493,.522,.655,.793,.873],
            'AMF':  [.511,.422,.348,.178,.516,.650,.682,.708],
            'LRao': [.572,.617,.546,.539,.673,.712,.768,.791]},
}
NGRID = [20,40,60,100,200,500,1000,2000]
res = json.load(open(OUT))
cells = {}
for k, v in res.items():
    if v['mode'] != MODE or v['front'] != FRONT: continue
    cells.setdefault(v['n'], []).append(v['pd_best'])
ns = sorted(cells)
mu = [np.mean(cells[n]) for n in ns]
sd = [np.std(cells[n]) for n in ns]
fig, ax = plt.subplots(figsize=(8, 5), dpi=120)
ax.errorbar(ns, mu, yerr=sd, color='#2a78d6', lw=2, marker='o', capsize=3,
            label=f'ours ({FRONT}, best-epoch)')
for name, color in (('DART', '#eb6834'), ('AMF', '#1baf7a'), ('LRao', '#eda100')):
    ax.plot(NGRID, PUB[MODE][name], color=color, lw=2, ls='--', label=f'{name} (published)')
ax.set_xscale('log'); ax.set_xlabel('n train'); ax.set_ylabel(f'Pd @ Pfa={PFA}')
ax.set_title(f'{MODE} n-sweep'); ax.grid(alpha=.25); ax.legend(frameon=False)
plt.show()

In [ ]:
# Zip and download results + checkpoints
import shutil, os
from google.colab import files
shutil.make_archive('iid_results', 'zip', '.', OUT)
files.download(OUT)
if os.path.isdir(CKPT_DIR) and os.listdir(CKPT_DIR):
    shutil.make_archive('iid_ckpts', 'zip', '.', CKPT_DIR)
    files.download('iid_ckpts.zip')